# Анализ видео

## Подготовка данных

In [ ]:
# импортируем библиотеку
import cv2

In [ ]:
# считываем видео целиком
video = cv2.VideoCapture('file_name.mp4')

In [ ]:
# получаем частоту кадров
fps = round(video.get(cv2.CAP_PROP_FPS))

# нам нужно записать по одному кадру в секунду, т.е. каждый n-ый кадр

# получаем число кадров во всем видео
frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT)) 

# длительность в секундах (в одной секунде fps(число) кадров)
duration = int(frame_count/fps) 

In [ ]:
# список для записи кадров
captures_array = []

In [ ]:
frame_count

In [ ]:
for i in range(frame_count):
    # покадровое считывание видео
    # ret: итог операции (True/False)
    # frame: кадр в формате массива rgb каждого пикселя
    ret, frame = video.read()

    if i % 100 == 0:
        print(i)
    
    # если номер кадра кратен числу кадров в секунду,добавляем его в список
    if i % fps == 0:
        captures_array.append(frame)

In [ ]:
print(len(captures_array))

## Обработка данных

In [ ]:
# для кластеризации
from scipy.cluster.vq import kmeans

# для работы с массивами
import numpy as np

# для работы с таблицами
import pandas as pd

In [ ]:
# функция сбора всей палитры фильма: на вход принимает список кадров и число центроидов
def palette_collector(captures, clusters_num):
    
    # список для записи палитр всех кадров
    dominant_colors = []
    
    i = 0
    
    for capture in captures:
    
        # определяем размеры массива
        shape = capture.shape

        # пересобираем в одномерный массив
        ar = capture.reshape(np.product(shape[:2]), shape[2]).astype(float)

        # находим заданное число центроидов
        centroids,_ = kmeans(ar, clusters_num)     
        
        # округляем RGB значения центроидов
        round_centroids = []

        for centroid in centroids:
            centroid = list(centroid)
            new_centroid = [round(centroid[0]), round(centroid[1]), round(centroid[2])]
            round_centroids.append(new_centroid)

        dominant_colors.append(round_centroids)
        
        i+=1

        if i % 10 == 0:
            print(f'{i}: done')
        
    return(dominant_colors)

In [ ]:
# применяем функцию к собраному ранее массиву кадров
film_palette = palette_collector(captures_array, 5)

# сохраняем данные в таблицу
film_df = pd.DataFrame({'Палитра' : film_palette})

film_df.to_excel('file_name.xlsx')

## Визуализация данных

In [ ]:
# для работы с таблицами
import pandas as pd

# для работы с массивами
import numpy as np

# для парсинга списков из таблицы xlsx
import ast

# для создания изображения
import cv2

In [ ]:
film_df = pd.read_excel('file_name.xlsx')

In [ ]:
# функция считывания данных из xlsx таблицы
def colors_collector(colors_df):

    rgb_array = []

    for pallete in colors_df['Палитра']:
        # считываем списки палитр с учетом скобок: []
        pallete = ast.literal_eval(pallete)
        for color in pallete:
            rgb_array.append(color)
    
    # возвращаем одномерный список с цветами
    return(rgb_array)

In [ ]:
# применяем функцию к таблице
film_palette = colors_collector(film_df)

In [ ]:
# функция для генерации изображения: на вход принимает список цветов и название для записи файла палитры

def generate_img(colors, file_name):
    colors_length = len(colors)
    
    # высота изображения (палитры) в пикселях
    height = 500;
    
    # создаем пустой массив
    # height: число строк
    # colors_length: число столбцов (длина палитры - число цветов в ней)
    # 3: массив одного цвета (r, g, b)
    img_array = np.zeros((height, colors_length, 3), np.uint8) 

    # заполняем пустой массив изображения каждым цветом
    for x in range(colors_length):
        img_array[:,x] = colors[x]
        
    # записываем данные о пикселях в файл
    cv2.imwrite(file_name, img_array)

In [ ]:
# генерируем изображение
generate_img(film_palette, 'file_name.png')

## Установка библиотек

In [ ]:
#!pip install opencv-python